# K-SPADA Binning Data Preparation

This tutorial preserves zero expression and maps nonzero values to cell-wise quantile bins for a generic AnnData object.

In [ ]:
from pathlib import Path
import anndata as ad
import numpy as np
import KSPADA

PROJECT_ROOT = Path("/your_path/K-SPADA")
DATA_DIR = Path("/your_path/example_dataset")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "binning"
SOURCE_H5AD = DATA_DIR / "source_data.h5ad"
OUTPUT_H5AD = OUTPUT_DIR / "binned_data.h5ad"
BIN_COUNT = 8
RANDOM_SEED = 1

## Step 1 — Check zero-preserving quantile binning

In [ ]:
example_expression = np.array(
    [
        [0.0, 0.2, 0.5, 1.0, 2.0, 4.0],
        [0.0, 0.0, 3.0, 3.0, 6.0, 9.0],
    ],
    dtype=np.float32,
)
example_binned = KSPADA.bin_expression_matrix(
    example_expression,
    bin_count=BIN_COUNT,
    random_seed=RANDOM_SEED,
)
assert np.all(example_binned[example_expression == 0] == 0)
assert example_binned.min() == 0
assert example_binned.max() <= BIN_COUNT - 1

## Step 2 — Bin the AnnData expression matrix

In [ ]:
source_adata = ad.read_h5ad(SOURCE_H5AD)
binned_adata = KSPADA.bin_adata(
    source_adata,
    bin_count=BIN_COUNT,
    random_seed=RANDOM_SEED,
    copy=True,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
binned_adata.write_h5ad(OUTPUT_H5AD)

## Notes

- Bin 0 represents zero detected expression.
- Bins 1 through 7 represent the relative rank of nonzero expression within each cell.